# Tradetropy - Quick Start

Welcome to **Tradetropy**, a fast backtesting and live-trading framework for
algorithmic strategies.

This notebook is the 5-minute tour. You will:

1. Load and **look at** market data (`TickData` / `KlineData`).
2. Write a simple **SMA crossover** strategy using `Signal`.
3. **Backtest** it on candles, read the **stats** and **trades**, and **plot** it.
4. Run the *same* strategy on a **tick** feed - no code changes.

Everything here runs offline on the bundled sample datasets, so you can
execute the notebook top to bottom without any downloads or API keys.

## Installation

```bash
pip install tradetropy
```

The base install already includes the backtesting engine, indicators and the
interactive [Bokeh](https://bokeh.org) chart used below.

In [ ]:
import numpy as np

from tradetropy import Strategy, BacktestEngine, TickData, KlineData
from tradetropy.ta import SMA
from tradetropy.signal import Signal
from tradetropy.datasets import load_btcusd_1m, load_mesu26_ticks

## 1. The data types

Tradetropy speaks two data types, and the *type* is what tells the engine how to
run - there are no mode flags:

- **`KlineData`** - OHLC candles, a `[N x 7]` matrix of
  `ts, open, high, low, close, volume, turnover`.
- **`TickData`** - raw trades/quotes, a `[N x 7]` matrix of
  `ts, bid, ask, volume, flags, volume_real, price`.

`tradetropy.datasets` bundles small, real samples of both so this notebook (and
the examples in the guide) runs with no downloads or API keys. In your own
work, load real data with `tradetropy.io` (`read_klines`, `read_ticks`, ...) or
build these arrays from your own data source.

In [ ]:
klines = load_btcusd_1m()      # 500 one-minute BTCUSDT candles
ticks = load_mesu26_ticks()    # 2000 Micro E-mini S&P 500 futures ticks

### Looking at the data

Both types render as a clean table in Jupyter (a head/tail preview, just like a
DataFrame). Put the object as the last expression of a cell to see it:

In [ ]:
klines

Grab just the first or last rows with `.head()` / `.tail()`:

In [ ]:
klines.tail(5)

In [ ]:
ticks.head(5)

Need a full pandas DataFrame (with a parsed `datetime` column) for your own
analysis or plotting? Use `.to_df()`:

In [ ]:
klines.to_df().head()

## 2. A strategy: SMA crossover

A strategy subclasses `Strategy`. Declare your data and indicators in `init()`,
and react to each new data point in `on_data()`. Orders go through the session
`self.sesh`.

We use `Signal` to detect the crossings cleanly:

- `Signal("partial")` compares the latest values `[-1]` vs `[-2]` - correct on
  **candles**, where each `on_data()` is already a closed bar.
- `Signal("closed")` shifts back one bar - recommended on **ticks**, so you only
  act on fully closed candles and never on a partial one.

The same class below picks the right mode automatically from `self.feed_type`,
so it runs unchanged on candles *and* ticks.

In [ ]:
class SmaCross(Strategy):
    """Long-only SMA crossover: buy on golden cross, exit on death cross."""

    fast_length = 10
    slow_length = 30

    def __init__(self, symbol):
        super().__init__()
        self.symbol = symbol

    def init(self):
        # 1-minute candles. In tick mode the engine builds these from ticks;
        # in kline mode it reads the candles you passed in.
        self.ohlc = self.subscribe_ohlc(
            self.symbol, timeframe='1m', window_size=200
        )
        # Two SMAs on the close price.
        self.fast = self.add_indicator(self.ohlc.close, SMA(self.fast_length))
        self.slow = self.add_indicator(self.ohlc.close, SMA(self.slow_length))
        # "closed" on ticks (avoid partial-bar signals), "partial" on candles.
        self.sig = Signal("closed" if self.feed_type == "tick" else "partial")

    def on_data(self):
        if self.sig.crossover(self.fast, self.slow):
            if not self.sesh.positions(self.symbol):
                self.sesh.buy(self.symbol, volume=1)
        elif self.sig.crossunder(self.fast, self.slow):
            for pos in self.sesh.positions(self.symbol):
                self.sesh.position_close(pos.ticket)

## 3. Backtest on candles

Prepare the engine with `by_klines`, then chain `run()`. If you don't pass a
session, Tradetropy uses a simulated broker with a \$10,000 starting balance
and no commission.

In [ ]:
bt = BacktestEngine.by_klines(SmaCross('BTCUSDT'), data=(klines,)).run()

### Performance stats

`bt.stats` is a full performance report:

In [ ]:
bt.stats

Every metric is also accessible by key:

In [ ]:
print("Return [%]     :", round(bt.stats["Return [%]"], 2))
print("Sharpe Ratio   :", round(bt.stats["Sharpe Ratio"], 2))
print("Max Drawdown[%]:", round(bt.stats["Max. Drawdown [%]"], 2))
print("# Trades       :", bt.stats["# Trades"])
print("Win Rate [%]   :", round(bt.stats["Win Rate [%]"], 2))

### The trades

`bt.stats.trades` is a DataFrame of every closed trade:

In [ ]:
bt.stats.trades.head()

And `bt.stats.equity_curve` is the equity over time as a pandas Series:

In [ ]:
bt.stats.equity_curve.tail()

## 4. Plot the backtest

`bt.plot()` renders an interactive Bokeh chart: candles, the SMA overlays, trade
markers and the equity curve. Use `output="notebook"` to draw it inline.

In [ ]:
bt.plot(output="notebook", theme="dark", plot_drawdown=True)

## 5. The same strategy on ticks

Nothing about `SmaCross` changes. We just hand the engine a `TickData` via
`by_ticks` instead. The engine aggregates the ticks into the 1-minute candles
the strategy subscribed to, and `Signal("closed")` kicks in automatically.

In [ ]:
bt_ticks = BacktestEngine.by_ticks(SmaCross('MESU26'), data=(ticks,)).run()
bt_ticks.stats

> The tick sample here is a few thousand prints, so it produces just a
> handful of trades - that's expected. The point is that the **same strategy
> code** runs on both candle and tick feeds.

## Where to go next

- **Multi-timeframe / multi-symbol** - subscribe to several `subscribe_ohlc`
  timeframes and symbols at once.
- **Optimization** - sweep `fast_length` / `slow_length` with
  `PoolBacktestEngine` (and feel the speed).
- **Data I/O** - `read_klines`, `save_ticks` across `.npz` (the base binary
  format), CSV, and the optional Parquet / HDF5 extras.
- **Backtest -> live** - the same `Strategy` runs under `LiveEngine` and
  `ReplayEngine`; see the live-trading guide.
- **Indicators & order flow** - `add_indicator`, `use_tool`, Volume Profile,
  `LargeTrades`, `DeepTrades`, and more.